## 1. Importar librerías

Framework: Pytorch

In [1]:
!pip install -q torchinfo
#!pip install -q torch torchvision torchinfo tensorboard scikit-learn seaborn matplotlib numpy pandas opencv-python pillow h5py scipy tqdm

In [2]:
# Librerías de Python
import os
import json
import csv
import sys
import time
import random
import warnings
from glob import glob
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple

# Manejo de datos y matemáticas
import numpy as np
import pandas as pd
import h5py

# Procesamiento de imágenes
import cv2
from PIL import Image
from scipy.ndimage import gaussian_filter

# Visualización y barras de progreso
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

# Machine Learning (Scikit-Learn)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Deep Learning: PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary

# Deep Learning: Torchvision (Computer Vision)
import torchvision
from torchvision import datasets
import torchvision.transforms as T
from torchvision.transforms import v2 as transforms
from torchvision.ops import Conv2dNormActivation
import torchvision.models as models

%matplotlib inline
warnings.filterwarnings("ignore")

Importación de funciones y clases ya definidas en el código de entrenamiento

In [ ]:
# En este caso, como se empleó Google Colab, se monta el drive para acceder a los archivos del proyecto
# En otro entorno, se debería ajustar la ruta de importación del código de entrenamiento

from google.colab import drive
#Para importar del código de entrenamiento En Google Colab
drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/Colab Notebooks')

from train_CSRNet_nwpu_b import (
    TrainingConfig,
    make_layers,
    CSRNet
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Descarga del dataset

Vamos a cargar las imágenes, que se encuentran divididas en varias carpetas. Necesitamos definir la estructura de directorios donde se alojarán los datos. Cada carpeta contiene 1000 imágens, excepto la última que contiene 1109 para completar las 5109 totales del dataset. El 70% de las imágenes están etiquetdas, las utilizaremos como conjunto para training y validation. El 30% de imágenes restantes serán empleadas para el conjunto de test en modo 'blind'.

| datos | nº imágenes | % |
| --- | --- | --- |
| training | 3609 | 70,64% |
| test | 1500 | 29,36% |

Dos modos de evaluación:


* **Blind:** Para evaluar sobre las 1500 imágenes de test se generan las predicciones y se suben a la plataforma web/API de los autores del dataset (https://www.crowdbenchmark.com/nwpucrowd.html). El servidor, que tiene las anotaciones ocultas, calcula las métricas.
*  **No Blind test:** Habría que dividir las 3609 imágenes públicas en nuestros propios subconjuntos de Train, Validation y Test. Luego la función test calcula el error (MAE/MSE) en este modo.

In [ ]:
!wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/EX3GjMQqlZROmVNeqGSnYh4B66K0VobqBc2EznW7xuFTWQ?e=3hgZLA&download=1" -O "nwpu_crowd_images_4.zip"
!wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/EVeif8uw4TtFgkGx0q1FKcEB9l1LKlQSXGl3MdCCDr_sjA?e=96L8KP&download=1" -O "nwpu_crowd_images_5.zip"
!wget "https://usales-my.sharepoint.com/:u:/g/personal/hrm_usal_es/EWf9gbkfUH5Jg7ixk9l763ABQ2smxVMK59QXzysE3Rgllg?e=bMwv1h&download=1" -O "nwpu_crowd_json.zip"

Estructura de directorios del entorno

In [7]:
!mkdir -p data/nwpu/images
!unzip -q "nwpu_crowd_images_4.zip" -d data/nwpu/images
!unzip -q "nwpu_crowd_images_5.zip" -d data/nwpu/images

!mkdir -p data/nwpu/json
!unzip -q "nwpu_crowd_json.zip" -d data/nwpu/json

In [8]:
%%bash

mkdir -p data/images/test

for i in $(seq -f "%04g" 3610 5109); do
  mv "data/nwpu/images/${i}.jpg" "data/images/test/" 2>/dev/null || true
done

In [ ]:
%%bash

mkdir -p data/images/no_blind_test

for i in $(seq -f "%04g" 3001 3609); do
  mv "data/nwpu/images/${i}.jpg" "data/images/no_blind_test/" 2>/dev/null || true
done

In [9]:
!echo "Blind Test:" $(ls data/images/test | wc -l) "imágenes"
!echo "No Blind Test:" $(ls data/images/no_blind_test | wc -l) "imágenes"

Blind Test: 1500 imágenes
ls: cannot access 'data/images/no_blind_test': No such file or directory
No Blind Test: 0 imágenes


Comprobaciones iniciales del dataset

In [10]:
test_root = os.path.join("data", "images", "test")

# Comprobamos dimensiones de alguna de las imágenes (p.ej. 10 primeras)
for i, file in enumerate(sorted(os.listdir(test_root))[:10]):
    path = os.path.join(test_root, file)
    with Image.open(path) as img:
        print(f"{file}: {img.size}  (ancho x alto)")

3610.jpg: (2302, 1535)  (ancho x alto)
3611.jpg: (6000, 4000)  (ancho x alto)
3612.jpg: (3264, 2448)  (ancho x alto)
3613.jpg: (2304, 1535)  (ancho x alto)
3614.jpg: (5184, 3456)  (ancho x alto)
3615.jpg: (4032, 3024)  (ancho x alto)
3616.jpg: (5184, 3456)  (ancho x alto)
3617.jpg: (5184, 3456)  (ancho x alto)
3618.jpg: (4032, 3024)  (ancho x alto)
3619.jpg: (4928, 3264)  (ancho x alto)


**Carga de anotaciones desde .json con el formato esperado**

In [11]:
def load_points_from_json_dir(folder_path):
       """
       Recorre una carpeta que contiene múltiples archivos JSON, cada uno con la estructura:
       {
              "img_id": "XXXX.jpg",
              "human_num": 0,
              "points": [[x, y], ...],
              "boxes": [[xmin, ymin, xmax, ymax], ...]

       Devuelve un diccionario:
       {
               "XXXX.jpg": [[x1, y1], [x2, y2], ...],
              ...
       }
       """
       folder = Path(folder_path)
       anotacion = {}

       for json_file in folder.glob("*.json"):
              with json_file.open('r', encoding='utf-8') as f:
                     data = json.load(f)

              img_id = data.get('img_id')
              points = [list(p) for p in data.get('points', [])]
              anotacion[img_id] = points

       return anotacion

In [12]:
#Carga de las anotaciones a partir de los .json
ann = load_points_from_json_dir("data/nwpu/json")

#Comprobación de las anotaciones
for idx, (img_id, points) in enumerate(ann.items()):
        if idx == 10:  #Ver solo los idx primeros
          break
        points_str = ",  ".join(f"({x},{y})" for x, y in points)
        print(f"[{img_id}] -> [{points_str}]")

[1418.jpg] -> [(659.6800000000001,2769.44),  (1.7280399333685634e-13,2699.52),  (133.76000000000016,2416.7999999999997),  (255.36000000000013,2243.52),  (9.120000000000172,2085.44),  (18.24000000000017,1951.68),  (486.4000000000001,2018.56),  (614.08,1966.88),  (407.3600000000001,2371.2000000000003),  (614.08,2340.8),  (741.7600000000001,2416.7999999999997),  (1027.52,2006.4),  (1197.76,2252.6400000000003),  (1428.8000000000002,3073.44),  (1550.3999999999999,2468.48),  (1337.6,2389.44),  (1471.3600000000001,2115.8399999999997),  (2006.3999999999999,2535.3599999999997),  (2048.96,3009.6),  (2389.4399999999996,3058.2400000000002),  (1711.52,2447.2),  (1650.7199999999998,2577.92),  (1659.8399999999997,2191.84),  (1866.56,2222.24),  (1878.7199999999996,2073.28),  (2207.0399999999995,2261.7599999999998),  (2307.3599999999997,2559.68),  (2526.24,2559.68),  (2854.5599999999995,2504.96),  (2663.0399999999995,2359.04),  (2583.9999999999995,2073.28),  (2553.6,1887.84),  (2073.2799999999997,1757.

**Reproducibilidad experimental**

El objetivo de la función *set_seed* es garantizar resultados consistentes utilizando los mismos datos y parámetros, reducir la variabilidad no explicada entre distintas ejecuciones de entrenamiento. Si se cambia un parámetro (p.ej. learning rate) de un entrenamiento respecto de otro debemos ser capaces de explicar el efecto de este sobre los resultados.

In [13]:
#Establecer semilla para que las ejecuciones sean reproducibles
def set_seed(seed):
    random.seed(seed)           #Módulos estándar de Python
    np.random.seed(seed)        #Operaciones de NumPy
    torch.manual_seed(seed)     #CPU y GPU por defecto

    if torch.cuda.is_available():
       torch.cuda.manual_seed(seed)
       torch.cuda.manual_seed_all(seed)    #Todas las GPUs en el sistema

       #Modo de búsqueda donde cuDNN prueba múltiples algoritmos para una operación dada y selecciona el más rápido para el tamaño de entrada específico
       torch.backends.cudnn.benchmark = False #False = Evita la selección dinámica de algoritmos
       #Obliga a la biblioteca a utilizar únicamente algoritmos que son deterministas
       torch.backends.cudnn.deterministic = True

set_seed(28)

##3. TEST

**Funciones Auxiliares**

In [14]:
DENSITY_BINS = [(0, 50), (50, 200), (200, 1e9)]

#Media y desviación previamente calculada
mean = [0.44722986, 0.41087792, 0.39603603]
std =  [0.26052581, 0.24937533, 0.2516997]

#normalización de VGG16
#mean=[0.485, 0.456, 0.406]
#std=[0.229, 0.224, 0.225]

def _create_density_from_points(points, H, W, sigma=1.0):
    """
    Genera un mapa de densidad para visualización.
    Se marca cada posición de cabeza con un píxel de valor 1 y se aplica un filtro Gaussiano (difuminado) a la imagen.
    Devuelve mapa (float32 numpy array (H, W))
    """
    density = np.zeros((H, W), dtype=np.float32)
    for p in points:
        x, y = p
        xi = int(round(x))
        yi = int(round(y))
        if 0 <= xi < W and 0 <= yi < H:
            density[yi, xi] += 1.0
    if sigma > 0:
        #mode='reflect' para evitar que la densidad caiga artificialmente a 0 en los bordes de la imagen
        density = gaussian_filter(density, sigma=sigma, mode='reflect')
    return density

def density_bucket(gt_count):
    """
    Clasificador de dificultad de la imagen según el número de personas.
    Asigna un índice basándose en el conteo total del Ground Truth.
    Sirve para analizar si el modelo falla más en multitudes dispersas o densas (Evaluación estratificada, por BINS).
    """
    for i, (low, high) in enumerate(DENSITY_BINS):
      if low <= gt_count < high:
          return i
    return len(DENSITY_BINS) - 1

def compute_game(pred_map, gt_map, level):
    """
    Calcula la métrica GAME (Grid Average Mean Absolute Error) que mide la precisión
    de la localización espacial del conteo dividiendo la imagen en una cuadrícula de 2^level x 2^level celdas, donde:

      - Level 0: MAE normal (imagen completa)
      - Level 1: Divide en 4 cuadrantes
      - Level 2: Divide en 16 cuadrantes
      - Level 3: Divide en 36 cuadrantes
    """
    if level == 0:
      return abs(pred_map.sum() - gt_map.sum())

    h, w = pred_map.shape
    splits = 2 ** level
    h_step, w_step = h // splits, w // splits

    error = 0.0
    for i in range(splits):
      for j in range(splits):
          #para cada celda
          p = pred_map[i*h_step:(i+1)*h_step, j*w_step:(j+1)*w_step].sum()
          g = gt_map[i*h_step:(i+1)*h_step, j*w_step:(j+1)*w_step].sum()
          #acumulación del error
          error += abs(p - g)
    return error

def denormalize_tensor(img_t: torch.Tensor, mean=mean, std=std) -> np.ndarray:
    """
    Invierte la normalización aplicada en el preprocesamiento para recuperar la imagen original (RGB en rango [0,1]).
    Realiza la operación inversa: pixel = (tensor * std) + mean.
    """
    img = img_t.detach().cpu().clone()
    for t, m, s in zip(img, mean, std):
      t.mul_(s).add_(m)
    arr = img.clamp(0,1).permute(1,2,0).numpy()
    return arr

**Función principal**

In [15]:
def test_csrnet(
    model: nn.Module,
    weights_path: str,
    images_dir: str,
    output_dir: str,
    image_size: int = 512,
    out_factor: int = 8,
    sigma: float = 50.0,
    min_sigma_out: float = 3.0,
    device: Optional[str] = None,
    blind: bool = False,
    submission_path: Optional[str] = None,
    annotations: Optional[Dict[str, List[Tuple[float,float]]]] = None,
    results_csv: Optional[str] = None,
    summary_csv: Optional[str] = None,
    save_visuals: bool = False
):

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(output_dir, exist_ok=True)

    visuals_dir = os.path.join(output_dir, "visualizations")
    if save_visuals:
        os.makedirs(visuals_dir, exist_ok=True)

    #Cargar pesos entrenados
    state = torch.load(weights_path, map_location=device)
    try:
        model.load_state_dict(state)
    except RuntimeError:
        model.load_state_dict(state, strict=False)

    #Transformaciones
    transform = T.Compose([
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std)
    ])

    img_paths = sorted(
        p for p in Path(images_dir).glob("*")
        if p.suffix.lower() in (".jpg", ".jpeg", ".png")
    )
    if len(img_paths) == 0:
        raise ValueError(f"No se han encontrado imágenes en {images_dir}")

    model.eval()
    model.to(device)

    results = []
    mae_errors, rmse_errors = [], []
    density_bin_errors = {i: [] for i in range(len(DENSITY_BINS))}
    game_errors = {0: [], 1: [], 2: [], 3: []}
    inference_times = []

    #Para las visualizaciones de ejemplo
    max_visuals = 10
    visual_count = 0

    with torch.no_grad():
        for img_path in tqdm(img_paths, desc="Testing"):
            img = Image.open(img_path).convert("RGB")
            orig_W, orig_H = img.size

            img_transf = transform(img)
            input_tensor = img_transf.unsqueeze(0).to(device)

            start_time = time.time()
            preds = model(input_tensor)
            if device == "cuda":
                torch.cuda.synchronize()
            inference_times.append(time.time() - start_time)

            preds = preds.squeeze().cpu().numpy()
            pred_count = float(preds.sum())

            #GT
            gt_map, gt_count = None, None
            if (not blind) and annotations is not None and img_path.name in annotations:
                points = annotations[img_path.name]

                out_H = max(1, image_size // out_factor)
                out_W = max(1, image_size // out_factor)

                scale_x = out_W / orig_W
                scale_y = out_H / orig_H
                scaled_pts = [(x * scale_x, y * scale_y) for x, y in points]

                sigma_out = max(min_sigma_out, sigma * 0.5 * (scale_x + scale_y))
                gt_map = _create_density_from_points(scaled_pts, out_H, out_W, sigma_out)
                gt_count = float(gt_map.sum())

            #####################
            ## VISUALIZACIONES ##
            #####################
            if save_visuals and visual_count < max_visuals:

                img_vis = denormalize_tensor(img_transf, mean=mean, std=std)

                ###### NO BLIND ######
                if not blind and gt_map is not None:
                    fig, axs = plt.subplots(1, 3, figsize=(12, 4))

                    axs[0].imshow(img_vis)
                    axs[0].set_title("Image")

                    axs[1].imshow(gt_map, cmap="jet")
                    axs[1].set_title(f"GT (count={gt_count:.1f})")

                    axs[2].imshow(preds, cmap="jet")
                    axs[2].set_title(f"Pred (count={pred_count:.1f})")

                ####### BLIND ######
                elif blind:
                    pred_map_resized = torch.tensor(preds)[None, None]
                    pred_map_resized = torch.nn.functional.interpolate(
                        pred_map_resized,
                        size=(image_size, image_size),
                        mode="bilinear",
                        align_corners=False
                    ).squeeze().numpy()

                    fig, axs = plt.subplots(1, 2, figsize=(10, 4))

                    axs[0].imshow(img_vis)
                    axs[0].set_title("Image")

                    axs[1].imshow(img_vis)
                    axs[1].imshow(pred_map_resized, cmap="jet", alpha=0.5)
                    axs[1].set_title(f"Predicted Density (count={pred_count:.1f})")

                else:
                    fig = None

                if fig is not None:
                    for ax in fig.axes:
                        ax.axis("off")
                    save_path = os.path.join(visuals_dir, f"{img_path.name}.png")
                    plt.savefig(save_path, bbox_inches="tight", pad_inches=0.1)
                    plt.close(fig)
                    visual_count += 1

            ############################
            ## MÉTRICAS (Si No Blind) ##
            ############################
            if not blind and gt_map is not None:
                error = pred_count - gt_count
                abs_error = abs(error)

                mae_errors.append(abs_error)
                rmse_errors.append(error ** 2)

                bin_id = density_bucket(gt_count)
                density_bin_errors[bin_id].append(abs_error)

                for L in game_errors:
                    game_errors[L].append(compute_game(preds, gt_map, L))

                results.append({
                    "image": img_path.name,
                    "gt_count": gt_count,
                    "pred_count": pred_count,
                    "error": error,
                    "abs_error": abs_error,
                    "density_bin": bin_id,
                    "game_0": game_errors[0][-1],
                    "game_1": game_errors[1][-1],
                    "game_2": game_errors[2][-1],
                    "game_3": game_errors[3][-1],
                })

            ######################
            ## MÉTRICAS (Blind) ##
            ######################
            if blind:
                results.append({
                    "image": img_path.name,
                    "gt_count": None,
                    "pred_count": pred_count
                })

    ################
    ## BLIND SUMMARY ##
    ################
    if blind:
        if submission_path is None:
            raise ValueError("En blind mode debes especificar submission_path")

        submission_path = os.path.join(output_dir, submission_path)
        with open(submission_path, "w") as f:
            for r in results:
                name = r["image"]
                clean_id = name.replace("img_", "").replace(".jpg", "").replace(".png", "")
                f.write(f"{clean_id} {int(round(r['pred_count']))}\n")

        summary = {
            "num_images": len(results),
            "mean_inference_time": float(np.mean(inference_times))
        }
        return results, summary

    ######################
    ## NO BLIND SUMMARY ##
    ######################
    summary = {
        "MAE": float(np.mean(mae_errors)),
        "RMSE": float(np.sqrt(np.mean(rmse_errors))),
        "mean_inference_time": float(np.mean(inference_times)),
        **{f"bin_{k}_mae": float(np.mean(v)) if v else None for k, v in density_bin_errors.items()},
        **{f"GAME_{k}": float(np.mean(v)) if v else None for k, v in game_errors.items()}
    }

    if results_csv:
        with open(os.path.join(output_dir, results_csv), "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=results[0].keys())
            writer.writeheader()
            writer.writerows(results)

    if summary_csv:
        with open(os.path.join(output_dir, summary_csv), "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=summary.keys())
            writer.writeheader()
            writer.writerow(summary)

    return results, summary


In [ ]:
model = CSRNet()
results, summary = test_csrnet(
    model=model,
    weights_path="/Best_model_47.pth",
    images_dir="data/images/no_blind_test",
    output_dir="output/test_results",
    annotations=ann,
    image_size=512,
    out_factor=8,
    sigma=40.0,
    min_sigma_out=2.0,
    blind=False,
    save_visuals=True,
    submission_path= "submission.txt",
    results_csv= "results.csv",
    summary_csv= "summary.csv",
)
print(summary)


Testing: 100%|██████████| 608/608 [04:32<00:00,  2.23it/s]

{'MAE': 125.68409139356817, 'RMSE': 918.7904766951133, 'mean_inference_time': 0.07026733654110055, 'bin_0_mae': 16.684598363563758, 'bin_1_mae': 28.371276770754065, 'bin_2_mae': 285.8596930099746, 'GAME_0': 125.68409729003906, 'GAME_1': 133.71311950683594, 'GAME_2': 145.03610229492188, 'GAME_3': 157.17987060546875}


In [16]:
model = CSRNet()
results, summary = test_csrnet(
    model=model,
    weights_path="Best_model_43.pth",
    images_dir="data/images/test",
    output_dir="output/test_results_blind",
    annotations=None,
    blind=True,
    submission_path="submission.txt",
    save_visuals=True,
    summary_csv= "summary.csv",
)
print(summary)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 81.3MB/s]
Testing: 100%|██████████| 1500/1500 [05:18<00:00,  4.71it/s]

{'num_images': 1500, 'mean_inference_time': 0.04615515915552775}
